# Twilight Struggle AI — Colab Training Notebook

This notebook trains the `TSBaselineModel` (factorized policy + value head) on
self-play Parquet data using a GPU runtime.

## Quick-start
1. **Runtime → Change runtime type → GPU** (T4 or A100).
2. Edit **Cell 1** (Config) — set `REPO_URL`, `GITHUB_TOKEN`, and `GDRIVE_PARQUET_PATH`.
3. **Runtime → Run all**.

## Data
Upload `filtered_v20.parquet` to your Google Drive (e.g. `MyDrive/ts-ai/`) before running.
Alternatively, skip Drive and use the file-upload fallback in Cell 3.

## Output
The best checkpoint is saved to `/content/checkpoints/baseline_best.pt`.
Cell 8 downloads it when you are done.

In [ ]:
# ============================================================
# Cell 1 — USER CONFIG: edit these values before running
# ============================================================

REPO_URL = "https://github.com/YOUR_USERNAME/twilight-struggle-ai.git"
GITHUB_TOKEN = ""  # personal access token for a private repo; leave empty if public
GDRIVE_PARQUET_PATH = "MyDrive/ts-ai/filtered_v20.parquet"  # path inside your Google Drive

# Training hyperparams
HIDDEN_DIM       = 256
EPOCHS           = 60
BATCH_SIZE       = 2048
LR               = 1.2e-3
WEIGHT_DECAY     = 1e-4
DROPOUT          = 0.1
LABEL_SMOOTHING  = 0.05
VALUE_TARGET     = "final_vp"    # "final_vp" or "winner_side"
VALUE_WEIGHT     = 1.0           # multiplier on value MSE loss
NUM_WORKERS      = 2             # DataLoader workers; set 0 if you hit OOM
CHECKPOINT_EVERY = 10            # save a per-epoch checkpoint every N epochs
SEED             = 42
VAL_FRACTION     = 0.10          # fraction of rows held out for validation

# Colab paths (no need to change)
REPO_LOCAL_DIR  = "/content/tsrl_repo"
DATA_DIR        = "/content/data"
CKPT_DIR        = "/content/checkpoints"
PARQUET_NAME    = "filtered_v20.parquet"

print("Config loaded.")

In [ ]:
# ============================================================
# Cell 2 — Install dependencies, clone repo, add to sys.path
# ============================================================

# polars / pyarrow / pydantic are not pre-installed on Colab; torch already is.
import subprocess, sys

print("Installing extra dependencies...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "polars>=0.20", "pyarrow>=14", "pydantic>=2.0"],
    check=True,
)
print("Dependencies installed.")

# Clone repo (or pull if already present from a previous run in the same session).
import os

if os.path.isdir(REPO_LOCAL_DIR):
    print(f"Repo already cloned at {REPO_LOCAL_DIR} — pulling latest...")
    subprocess.run(["git", "-C", REPO_LOCAL_DIR, "pull", "--ff-only"], check=True)
else:
    clone_url = REPO_URL
    if GITHUB_TOKEN:
        # Inject token for private repos: https://TOKEN@github.com/...
        clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", clone_url, REPO_LOCAL_DIR], check=True)
    print("Clone complete.")

# Make the project's Python packages importable.
python_dir = os.path.join(REPO_LOCAL_DIR, "python")
if python_dir not in sys.path:
    sys.path.insert(0, python_dir)
print(f"sys.path: {python_dir} added.")

In [ ]:
# ============================================================
# Cell 3 — Mount Google Drive and locate the parquet file
#           Falls back to direct file upload if Drive is skipped.
# ============================================================

import os

os.makedirs(DATA_DIR, exist_ok=True)
dest_path = os.path.join(DATA_DIR, PARQUET_NAME)

# --- Try Google Drive first ---
_use_drive = True
try:
    from google.colab import drive as _drive
    _drive.mount("/content/drive")
    gdrive_path = f"/content/drive/{GDRIVE_PARQUET_PATH}"
    if os.path.exists(gdrive_path):
        if not os.path.exists(dest_path):
            os.symlink(gdrive_path, dest_path)
        print(f"Data file via Google Drive: {gdrive_path}")
        print(f"  Size: {os.path.getsize(gdrive_path) / 1e6:.1f} MB")
        _use_drive = True
    else:
        print(f"WARNING: Parquet not found at {gdrive_path}")
        _use_drive = False
except Exception as _e:
    print(f"Google Drive not available ({_e}). Falling back to direct upload.")
    _use_drive = False

# --- Fallback: upload the file directly ---
if not _use_drive and not os.path.exists(dest_path):
    print("Please upload filtered_v20.parquet using the dialog below:")
    from google.colab import files as _files
    _uploaded = _files.upload()
    for _fname, _data in _uploaded.items():
        with open(dest_path, "wb") as _f:
            _f.write(_data)
        print(f"Saved uploaded file to {dest_path} ({len(_data)/1e6:.1f} MB)")
        break  # only expect one file

assert os.path.exists(dest_path), (
    f"Parquet file not found at {dest_path}.\n"
    "Either mount Google Drive with the correct path or upload the file above."
)
print(f"Ready: {dest_path} ({os.path.getsize(dest_path)/1e6:.1f} MB)")

In [ ]:
# ============================================================
# Cell 4 — Load dataset and show stats
# ============================================================

import time
import torch
from torch.utils.data import DataLoader
from tsrl.policies.dataset import TS_SelfPlayDataset

print(f"Loading dataset from {DATA_DIR} (value_target_mode={VALUE_TARGET!r}) ...")
t0 = time.time()
full_ds = TS_SelfPlayDataset(DATA_DIR, value_target_mode=VALUE_TARGET)
print(f"Dataset: {len(full_ds):,} rows loaded in {time.time() - t0:.1f}s")

# Quick sanity check: peek at one batch to confirm shapes.
# Use num_workers=0 here to avoid forking before the dataset is split.
_probe_loader = DataLoader(full_ds, batch_size=min(BATCH_SIZE, len(full_ds)), shuffle=False, num_workers=0)
_batch = next(iter(_probe_loader))
print("Batch tensor shapes:")
for _k, _v in sorted(_batch.items()):
    print(f"  {_k:25s}: {tuple(_v.shape)}  dtype={_v.dtype}")
del _probe_loader, _batch

In [ ]:
# ============================================================
# Cell 5 — Build model and show parameter count
# ============================================================

import torch
from tsrl.policies.model import TSBaselineModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

model = TSBaselineModel(dropout=DROPOUT, hidden_dim=HIDDEN_DIM).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: TSBaselineModel(hidden_dim={HIDDEN_DIM}, dropout={DROPOUT})")
print(f"  Trainable parameters: {n_params:,}")

In [ ]:
# ============================================================
# Cell 6 — Throughput benchmark (50 forward+backward batches)
#           Run this before the full training loop to get a
#           baseline samples/sec estimate for your runtime.
# ============================================================

import time
import torch
from torch.utils.data import DataLoader

# Use a small subset so the benchmark completes quickly.
N_BENCH = 50
_bench_loader = DataLoader(
    full_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    # Use forkserver context to avoid Polars/OpenMP deadlocks in forked workers.
    multiprocessing_context="forkserver" if NUM_WORKERS > 0 else None,
)

_opt_bench = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
model.train()

# Warm-up one batch before timing.
_batch0 = next(iter(_bench_loader))
_out0 = model(
    _batch0["influence"].to(device),
    _batch0["cards"].to(device),
    _batch0["scalars"].to(device),
)
_dummy = _out0["card_logits"].sum() * 0.0
_dummy.backward()
_opt_bench.zero_grad()

if device.type == "cuda":
    torch.cuda.synchronize()

t0 = time.time()
for _i, _batch in enumerate(_bench_loader):
    if _i >= N_BENCH:
        break
    _inf  = _batch["influence"].to(device)
    _crd  = _batch["cards"].to(device)
    _scl  = _batch["scalars"].to(device)
    _out  = model(_inf, _crd, _scl)
    # Dummy loss — we just want to measure data+forward+backward throughput.
    _loss = _out["card_logits"].sum() * 0.0
    _loss.backward()
    _opt_bench.zero_grad()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed = time.time() - t0
samples_per_sec = N_BENCH * BATCH_SIZE / elapsed
ms_per_batch = elapsed / N_BENCH * 1000
steps_per_epoch = (len(full_ds) * (1 - VAL_FRACTION)) / BATCH_SIZE
eta_minutes = steps_per_epoch * ms_per_batch / 1000 / 60

print(f"Throughput  : {samples_per_sec:,.0f} samples/s")
print(f"ms/batch    : {ms_per_batch:.1f} ms")
print(f"Est. time/epoch: {eta_minutes:.1f} min  ({EPOCHS} epochs -> {eta_minutes*EPOCHS:.0f} min)")

del _bench_loader, _opt_bench, _batch0, _out0, _dummy

In [ ]:
# ============================================================
# Cell 7 — Full training loop with live matplotlib curves
# ============================================================

%matplotlib inline

import os
import time
import random
import math

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
from IPython import display as _ipython_display

from tsrl.policies.model import TSBaselineModel
from tsrl.policies.dataset import TS_SelfPlayDataset

# ---- reproducibility ----
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---- train/val split by index (90/10) ----
n_total = len(full_ds)
n_val   = max(1, int(n_total * VAL_FRACTION))
n_train = n_total - n_val
# Contiguous split (no shuffle by game-id; acceptable for a simplified notebook).
train_ds = Subset(full_ds, range(n_train))
val_ds   = Subset(full_ds, range(n_train, n_total))
print(f"Dataset split: {n_train:,} train | {n_val:,} val")

# ---- dataloaders ----
# Use forkserver to avoid Polars/OpenMP thread-pool deadlocks in forked workers.
_mp_ctx = "forkserver" if NUM_WORKERS > 0 else None
_pin    = (device.type == "cuda")

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=_pin,
    drop_last=False, persistent_workers=(NUM_WORKERS > 0),
    multiprocessing_context=_mp_ctx,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=_pin,
    drop_last=False, persistent_workers=(NUM_WORKERS > 0),
    multiprocessing_context=_mp_ctx,
)

# ---- model ----
model = TSBaselineModel(dropout=DROPOUT, hidden_dim=HIDDEN_DIM).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {n_params:,} parameters  (hidden_dim={HIDDEN_DIM})")

# ---- optimizer + OneCycleLR scheduler ----
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    epochs=EPOCHS,
    steps_per_epoch=math.ceil(n_train / BATCH_SIZE),
    pct_start=0.10,          # 10% of total steps for linear warmup
    anneal_strategy="cos",
)

# ---- loss functions ----
ce_fn      = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
mode_ce_fn = nn.CrossEntropyLoss(ignore_index=-1, label_smoothing=LABEL_SMOOTHING)
mse_fn     = nn.MSELoss()

# ---- checkpoint directory ----
os.makedirs(CKPT_DIR, exist_ok=True)
best_ckpt_path = os.path.join(CKPT_DIR, "baseline_best.pt")

# ---- helper: top-1 accuracy ----
def _accuracy(logits, targets, ignore_index=-100):
    mask = targets != ignore_index
    if not mask.any():
        return float("nan")
    return (logits.argmax(dim=-1)[mask] == targets[mask]).float().mean().item()

# ---- helper: one epoch ----
def run_epoch(loader, is_train):
    model.train(is_train)
    tot_loss = tot_card = tot_mode = tot_country = tot_value = 0.0
    tot_card_acc = tot_mode_acc = tot_country_top1 = tot_val_mse = 0.0
    n = 0
    n_samples = 0
    t0 = time.time()

    ctx = torch.no_grad() if not is_train else torch.enable_grad()
    with ctx:
        for batch in loader:
            inf   = batch["influence"].to(device)
            crd   = batch["cards"].to(device)
            scl   = batch["scalars"].to(device)
            c_tgt = batch["card_target"].to(device)
            m_tgt = batch["mode_target"].to(device)
            co_tgt = batch["country_ops_target"].to(device)
            v_tgt = batch["value_target"].to(device)

            out = model(inf, crd, scl)
            card_logits            = out["card_logits"]
            mode_logits            = out["mode_logits"]
            country_logits         = out["country_logits"]
            country_strategy_logits = out["country_strategy_logits"]
            strategy_logits        = out["strategy_logits"]
            value_pred             = out["value"]

            card_loss  = ce_fn(card_logits, c_tgt)
            mode_loss  = mode_ce_fn(mode_logits, m_tgt)
            value_loss = mse_fn(value_pred, v_tgt)

            # Country loss: ops-weighted log-mixture CE, skipping SPACE/EVENT rows.
            co_mask = co_tgt.sum(dim=1) > 0
            country_top1 = 0.0
            if co_mask.any():
                ops_t    = co_tgt[co_mask]                              # (M, 86)
                ops_prob = ops_t / ops_t.sum(dim=1, keepdim=True)       # normalise
                mixing   = torch.softmax(strategy_logits[co_mask], dim=1)  # (M, 4)
                strat_p  = torch.softmax(country_strategy_logits[co_mask], dim=2)  # (M, 4, 86)
                mix_prob = (mixing.unsqueeze(2) * strat_p).sum(dim=1)  # (M, 86)
                country_loss = -(ops_prob * torch.log(mix_prob + 1e-8)).sum(dim=1).mean()
                country_top1 = (
                    country_logits[co_mask].argmax(dim=1)
                    == co_tgt[co_mask].argmax(dim=1)
                ).float().mean().item()
            else:
                country_loss = torch.tensor(0.0, device=device)

            loss = card_loss + mode_loss + country_loss + VALUE_WEIGHT * value_loss

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                scheduler.step()

            bs = inf.size(0)
            tot_loss     += loss.item()
            tot_card     += card_loss.item()
            tot_mode     += mode_loss.item()
            tot_country  += country_loss.item()
            tot_value    += value_loss.item()
            tot_card_acc    += _accuracy(card_logits, c_tgt)
            tot_mode_acc    += _accuracy(mode_logits, m_tgt, ignore_index=-1)
            tot_country_top1 += country_top1
            tot_val_mse     += value_loss.item()
            n_samples += bs
            n += 1

    elapsed = time.time() - t0
    throughput = n_samples / elapsed
    if n == 0:
        return {}, 0.0
    return {
        "loss":         tot_loss / n,
        "card_loss":    tot_card / n,
        "mode_loss":    tot_mode / n,
        "country_loss": tot_country / n,
        "value_loss":   tot_value / n,
        "card_top1":    tot_card_acc / n,
        "mode_acc":     tot_mode_acc / n,
        "country_top1": tot_country_top1 / n,
        "value_mse":    tot_val_mse / n,
    }, throughput

# ---- training state ----
history = {
    "epoch": [],
    "train_loss": [], "val_loss": [],
    "val_card_top1": [], "val_mode_acc": [],
    "val_country_top1": [], "val_value_mse": [],
    "train_throughput": [],
}
best_val_loss = float("inf")

# ---- live-plot helper ----
def _update_plot(history):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    ax = axes[0]
    ax.plot(history["epoch"], history["train_loss"], label="train")
    ax.plot(history["epoch"], history["val_loss"],   label="val")
    ax.set_title("Total loss")
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True)

    ax = axes[1]
    ax.plot(history["epoch"], history["val_card_top1"],    label="card top-1")
    ax.plot(history["epoch"], history["val_mode_acc"],     label="mode acc")
    ax.plot(history["epoch"], history["val_country_top1"], label="country top-1")
    ax.set_title("Policy accuracy (val)")
    ax.set_xlabel("Epoch")
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(True)

    ax = axes[2]
    ax.plot(history["epoch"], history["val_value_mse"], color="tab:orange", label="val value MSE")
    ax.set_title("Value MSE (val)")
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True)

    plt.tight_layout()
    _ipython_display.clear_output(wait=True)
    plt.show()

# ---- main training loop ----
print(f"Starting training: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LR}, device={device}")
t_total_start = time.time()

for epoch in range(1, EPOCHS + 1):
    t_epoch = time.time()

    train_metrics, train_tput = run_epoch(train_loader, is_train=True)
    val_metrics,   _          = run_epoch(val_loader,   is_train=False)

    val_loss = val_metrics.get("loss", float("nan"))
    is_best  = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss

    # Accumulate history for live plots.
    history["epoch"].append(epoch)
    history["train_loss"].append(train_metrics.get("loss", float("nan")))
    history["val_loss"].append(val_metrics.get("loss", float("nan")))
    history["val_card_top1"].append(val_metrics.get("card_top1", float("nan")))
    history["val_mode_acc"].append(val_metrics.get("mode_acc", float("nan")))
    history["val_country_top1"].append(val_metrics.get("country_top1", float("nan")))
    history["val_value_mse"].append(val_metrics.get("value_mse", float("nan")))
    history["train_throughput"].append(train_tput)

    epoch_sec = time.time() - t_epoch

    # Refresh live plot after every epoch.
    _update_plot(history)

    # Console summary line printed below the plot.
    best_tag = "  [BEST]" if is_best else ""
    print(
        f"Epoch {epoch:3d}/{EPOCHS}"
        f"  train_loss={train_metrics.get('loss', float('nan')):.4f}"
        f"  val_loss={val_loss:.4f}"
        f"  card_top1={val_metrics.get('card_top1', float('nan')):.3f}"
        f"  mode_acc={val_metrics.get('mode_acc', float('nan')):.3f}"
        f"  country_top1={val_metrics.get('country_top1', float('nan')):.3f}"
        f"  val_mse={val_metrics.get('value_mse', float('nan')):.4f}"
        f"  {train_tput:,.0f} samp/s"
        f"  {epoch_sec:.0f}s"
        + best_tag
    )

    # Save best-validation checkpoint.
    if is_best:
        ckpt = {
            "epoch":                epoch,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "train_metrics":        train_metrics,
            "val_metrics":          val_metrics,
            "config": {
                "hidden_dim":      HIDDEN_DIM,
                "dropout":         DROPOUT,
                "lr":              LR,
                "weight_decay":    WEIGHT_DECAY,
                "label_smoothing": LABEL_SMOOTHING,
                "value_target":    VALUE_TARGET,
                "epochs":          EPOCHS,
                "batch_size":      BATCH_SIZE,
                "seed":            SEED,
            },
        }
        torch.save(ckpt, best_ckpt_path)
        print(f"  -> Saved best checkpoint: {best_ckpt_path}")

    # Save periodic per-epoch checkpoint.
    if epoch % CHECKPOINT_EVERY == 0:
        epoch_ckpt_path = os.path.join(CKPT_DIR, f"baseline_epoch{epoch:03d}.pt")
        torch.save({
            "epoch":                epoch,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_metrics":          val_metrics,
        }, epoch_ckpt_path)
        print(f"  -> Saved epoch checkpoint: {epoch_ckpt_path}")

total_min = (time.time() - t_total_start) / 60
print(f"\nTraining complete in {total_min:.1f} min.")
print(f"Best val_loss = {best_val_loss:.4f}  -> {best_ckpt_path}")

In [ ]:
# ============================================================
# Cell 8 — Download best checkpoint
# ============================================================

import os
from google.colab import files

if os.path.exists(best_ckpt_path):
    print(f"Downloading {best_ckpt_path} ...")
    files.download(best_ckpt_path)
else:
    print(f"No best checkpoint found at {best_ckpt_path}.")
    print("Has training completed? Re-run Cell 7 first.")

In [ ]:
# ============================================================
# Cell 9 — Quick eval summary
#           Loads the best checkpoint and prints a final
#           metrics table.  Safe to run independently.
# ============================================================

import os
import torch

if not os.path.exists(best_ckpt_path):
    print(f"Checkpoint not found: {best_ckpt_path}")
    print("Run Cell 7 (training) first.")
else:
    ckpt = torch.load(best_ckpt_path, map_location="cpu", weights_only=False)
    best_epoch    = ckpt["epoch"]
    val_m         = ckpt.get("val_metrics", {})
    train_m       = ckpt.get("train_metrics", {})
    cfg           = ckpt.get("config", {})

    print("=" * 55)
    print(f"  Best checkpoint  —  epoch {best_epoch} / {cfg.get('epochs', '?')}")
    print("=" * 55)
    print(f"  {'Metric':<30} {'Train':>10} {'Val':>10}")
    print(f"  {'-'*50}")

    rows = [
        ("Total loss",        "loss"),
        ("Card loss",         "card_loss"),
        ("Mode loss",         "mode_loss"),
        ("Country loss",      "country_loss"),
        ("Value loss (MSE)",  "value_loss"),
        ("Card top-1 acc",    "card_top1"),
        ("Mode accuracy",     "mode_acc"),
        ("Country top-1",     "country_top1"),
        ("Value MSE",         "value_mse"),
    ]

    for label, key in rows:
        tr = train_m.get(key, float("nan"))
        vl = val_m.get(key, float("nan"))
        print(f"  {label:<30} {tr:>10.4f} {vl:>10.4f}")

    print("="*55)
    print("Config:")
    for k, v in cfg.items():
        print(f"  {k}: {v}")

    # If training history is still in memory, show the final plot too.
    if 'history' in dir() and history["epoch"]:
        _update_plot(history)